# Controlador multisemilla — Experimento 6, variante BORUTA TRADICIONAL (grupo A)

Corre `z635_WorkFlow_01_senior_boruta_tradicional.ipynb` (la senior con Boruta tradicional, paquete `Boruta` + `ranger`, en lugar de Canaritos) **una vez por semilla, una detrás de otra**, en segundo plano.
Ya viene configurado: variante `boruta_trad`, experimentos 6501 a 6505, semillas 123443, 444469, 987631, 467869, 763493.

Usa números de experimento distintos a la versión anterior con LightGBM (6401..6405) para no retomar sus `HT.RDATA` ni sus `modelos/`.

**Uso**
1. En CONFIG revisá solo `notebook_base`: tiene que ser la ruta donde subiste la notebook de Boruta tradicional.
2. Ejecutá todas las celdas (Run All). Después podés cerrar el navegador.
3. Para ver el avance corré **MONITOR** (antes CONFIG y FUNCIONES si se reinició el kernel). No vuelvas a correr GENERAR ni LANZAR mientras haya una corrida en curso.

Boruta escribe en el log una línea por ronda: `ver_log(123443)`. Esa etapa tarda varias horas por semilla.

In [1]:
# ================== CONFIGURACION (lo unico que se toca) ==================
CONFIG <- list(
  # nombre corto de la variante (se usa en carpetas y en el mensaje de Kaggle)
  variante = "boruta_trad",

  # las 5 semillas del experimento (MISMA lista y MISMO orden en todas las variantes)
  semillas = c(109843, 109847, 109849, 109859, 109873),

  # numero de experimento de la primera semilla; las demas son +1, +2, ...
  # baseline 6101.., desvios1 6201.., desvios3 6301.., boruta LightGBM 6401.., boruta tradicional 6501..
  experimento_inicial = 9540L,

  # NULL = correr todas las semillas en esta VM.
  # Si repartis en varias VMs, en cada una pone las que le tocan, ej: c(123443, 444469)
  # (el numero de experimento de cada semilla NO cambia)
  correr_solo = NULL,

  # parametros de la notebook base que se pisan en esta variante
  # (cualquier linea "PARAM$xxx <- valor" o "PARAM$xxx = valor" de la base)
  # parametros de Boruta tradicional (defaults del paquete: 100 rondas, p 0.01, 500 arboles)
  param_override = list(
    "PARAM$boruta$max_runs"  = 100,     # rondas maximas
    "PARAM$boruta$alpha"     = 0.01,   # p-valor del test (con Bonferroni)
    "PARAM$boruta$num_trees" = 500,    # arboles de ranger por ronda
    "PARAM$boruta$importancia" = "gini", # "gini" (rapida) o "permutacion" (default del paquete, ~45 min por ronda)
    "PARAM$boruta$rough_fix" = TRUE    # TentativeRoughFix sobre los tentativos que queden
  ),

  # TRUE = cada corrida sube sus 9 archivos a Kaggle (como la base)
  # FALSE = solo genera los csv en kaggle/ y los subis despues
  kaggle_submit = TRUE,

  # notebook base (workflow senior con Boruta tradicional, sin tocar)
  notebook_base = "/content/buckets/b1/labo2026v/ExpColaborativos/z635_WorkFlow_01_senior_boruta_tradicional_1.ipynb",

  # logs en disco LOCAL (el bucket no muestra los logs mientras se escriben);
  # al terminar cada corrida se copian a carpeta_bucket
  carpeta_local  = "~/exp6_control",
  carpeta_bucket = "/content/buckets/b1/exp/exp6_control",

  # cuantas corridas a la vez; se recorta solo si no alcanza la RAM
  max_paralelo       = 1,     # 1 = una detras de otra
  ram_por_corrida_gb = 110,   # pico de RAM de UNA corrida (ajustalo con lo que mida la 1ra)
  forzar_paralelo    = FALSE  # TRUE = ignorar el control de RAM (bajo tu riesgo)
)

### Funciones (no hace falta tocar)

In [2]:
if (!require("jsonlite")) install.packages("jsonlite")
require("jsonlite")
if (!require("data.table")) install.packages("data.table")
require("data.table")

escapar_regex <- function(x) gsub("([.$^*+?()|\\[\\]{}\\\\])", "\\\\\\1", x, perl = TRUE)

# reemplaza TODAS las asignaciones "PARAM$x <- ..." / "PARAM$x = ..." (inicio de linea)
pisar_parametro <- function(src, path, valor) {
  patron <- paste0("(?m)^([ \\t]*)", escapar_regex(path), "[ \\t]*(<-|=)[^\\n]*$")
  if (!grepl(patron, src, perl = TRUE))
    stop("No encontre en la notebook base la asignacion de ", path,
         " (revisa el nombre en CONFIG$param_override)", call. = FALSE)
  valor_txt <- paste(deparse(valor), collapse = "")
  gsub(patron, paste0("\\1", path, " <- ", valor_txt, "  # [controlador]"), src, perl = TRUE)
}

reemplazo_fijo <- function(src, viejo, nuevo, aviso) {
  if (grepl(viejo, src, fixed = TRUE)) return(gsub(viejo, nuevo, src, fixed = TRUE))
  warning(aviso, call. = FALSE)
  src
}

nombre_corrida <- function(experimento, semilla) paste0("WF", experimento, "_s", semilla)

# imprime hora + RAM actual y pico del proceso (lee /proc/self/status)
LINEA_RAM <- paste0(
  "local({ st <- readLines('/proc/self/status'); ",
  "g <- function(k) round(as.numeric(gsub('[^0-9]', '', grep(paste0('^', k), st, value = TRUE))) / 1024^2, 1); ",
  "paste0('ram=', g('VmRSS'), 'GB pico=', g('VmHWM'), 'GB') })")

generar_script <- function(semilla, experimento, cfg, dir_scripts) {
  nb <- fromJSON(cfg$notebook_base, simplifyVector = FALSE)
  partes <- c()
  for (i in seq_along(nb$cells)) {
    cell <- nb$cells[[i]]
    if (!identical(cell$cell_type, "code")) next
    src <- paste(unlist(cell$source), collapse = "")
    if (!nzchar(trimws(src))) next
    partes <- c(partes, paste0(
      "cat('\\n[celda ", i, "/", length(nb$cells), "] ', format(Sys.time(), '%Y-%m-%d %H:%M:%S'), ' ', ",
      LINEA_RAM, ", '\\n')\n", src, "\n"))
  }
  src <- paste(partes, collapse = "\n")

  overrides <- c(list("PARAM$semilla_primigenia" = semilla,
                      "PARAM$experimento" = experimento),
                 cfg$param_override)
  for (p in names(overrides)) src <- pisar_parametro(src, p, overrides[[p]])

  # el mensaje de Kaggle lleva variante y experimento, para despues separar los resultados
  src <- reemplazo_fijo(src, "\"-m 'envios=\", envios,",
    paste0("\"-m 'variante=", cfg$variante, " exp=\", PARAM$experimento, \" envios=\", envios,"),
    "No encontre el armado del mensaje de Kaggle; los submits no van a llevar la variante.")

  if (!isTRUE(cfg$kaggle_submit)) {
    src <- reemplazo_fijo(src, "salida <- system(linea, intern=TRUE)",
      "salida <- paste('[controlador] submit desactivado:', archivo_kaggle)",
      "No encontre la linea del submit a Kaggle; no pude desactivarlo.")
    src <- reemplazo_fijo(src, "Sys.sleep(30)", "# Sys.sleep(30) [controlador]",
      "No encontre el Sys.sleep(30) del submit.")
  }

  encabezado <- paste0(
    "# Script generado por el controlador multisemilla - NO editar a mano\n",
    "# variante=", cfg$variante, " semilla=", semilla, " experimento=", experimento, "\n",
    "# generado: ", format(Sys.time(), "%Y-%m-%d %H:%M:%S"), "\n",
    "cat('[INICIO] variante=", cfg$variante, " semilla=", semilla, " experimento=", experimento,
    " ', format(Sys.time(), '%Y-%m-%d %H:%M:%S'), '\\n')\n",
    "try(data.table::setDTthreads(as.integer(Sys.getenv('OMP_NUM_THREADS', '0'))), silent = TRUE)\n\n")
  pie <- paste0("\ncat('\\n###FIN_OK### ', format(Sys.time(), '%Y-%m-%d %H:%M:%S'), ' ', ", LINEA_RAM, ", '\\n')\n")

  arch <- file.path(dir_scripts, paste0(nombre_corrida(experimento, semilla), ".R"))
  writeLines(paste0(encabezado, src, pie), arch, useBytes = TRUE)
  arch
}

# ---------- monitoreo ----------
leer_log <- function(arch) if (file.exists(arch)) readLines(arch, warn = FALSE) else character(0)

procesos <- function(patron) {
  ps <- trimws(system("ps -eo pid=,rss=,args=", intern = TRUE))
  campos <- strsplit(ps, " +")
  pid  <- as.integer(sapply(campos, `[`, 1))
  rss  <- as.numeric(sapply(campos, `[`, 2))
  args <- sub("^[0-9]+ +[0-9]+ +", "", ps)
  ok <- grepl(patron, args, fixed = TRUE) & pid != Sys.getpid()
  data.table(pid = pid[ok], ram_gb = round(rss[ok] / 1024^2, 1))
}

dir_variante <- function(cfg = CONFIG) file.path(path.expand(cfg$carpeta_local), cfg$variante)

estado <- function(cfg = CONFIG) {
  man <- fread(file.path(dir_variante(cfg), "corridas.csv"))
  res <- rbindlist(lapply(seq_len(nrow(man)), function(k) {
    r <- man[k]
    lg <- leer_log(r$log)
    celdas <- grep("^\\[celda ", lg, value = TRUE)
    ok   <- any(grepl("###FIN_OK###", lg, fixed = TRUE))
    err  <- any(grepl("Execution halted|Ejecuci.n interrumpida", lg))
    pr   <- procesos(paste0("--file=", r$script))
    est <- if (ok) "OK" else if (nrow(pr)) "corriendo" else if (err) "ERROR" else if (length(lg)) "detenido" else "pendiente"
    data.table(experimento = r$experimento, semilla = r$semilla, estado = est,
               ram_gb = if (nrow(pr)) sum(pr$ram_gb) else NA_real_,
               ultima_celda = if (length(celdas)) sub("^(\\[celda [0-9/]+\\]).*?(ram=.*?) *$", "\\1 \\2", tail(celdas, 1), perl = TRUE) else "",
               min_sin_escribir = if (file.exists(r$log)) round(as.numeric(difftime(Sys.time(), file.mtime(r$log), units = "mins")), 1) else NA_real_)
  }))
  mem <- system("free -g | awk '/Mem:/ {print $2, $3, $7}'", intern = TRUE)
  mem <- as.numeric(strsplit(mem, " ")[[1]])
  cat(format(Sys.time(), "%H:%M:%S"), "| RAM VM: usada", mem[2], "GB de", mem[1], "GB, disponible", mem[3], "GB\n")
  res
}

ver_log <- function(sem, n = 40, cfg = CONFIG) {
  man <- fread(file.path(dir_variante(cfg), "corridas.csv"))
  arch <- man[semilla == sem, log]
  if (!length(arch)) stop("La semilla ", sem, " no esta en esta variante (o no corre en esta VM)")
  cat(tail(leer_log(arch), n), sep = "\n")
}

detener_todo <- function(cfg = CONFIG) {
  pids <- procesos(normalizePath(dir_variante(cfg), mustWork = FALSE))$pid
  if (length(pids)) tools::pskill(pids)
  cat("Detenidos", length(pids), "procesos de la variante", cfg$variante, "\n")
}

Loading required package: jsonlite

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%




### Recursos de la VM
Decide cuántas corridas entran a la vez según la RAM y reparte los núcleos entre ellas.

In [3]:
ram_total_gb <- as.numeric(system("awk '/MemTotal/ {print $2}' /proc/meminfo", intern = TRUE)) / 1024^2
nucleos <- parallel::detectCores()
semillas_vm <- if (is.null(CONFIG$correr_solo)) CONFIG$semillas else CONFIG$correr_solo
if (!all(semillas_vm %in% CONFIG$semillas)) stop("correr_solo tiene semillas que no estan en CONFIG$semillas")

max_por_ram <- max(1L, floor(ram_total_gb / CONFIG$ram_por_corrida_gb))
paralelo <- if (CONFIG$forzar_paralelo) CONFIG$max_paralelo else min(CONFIG$max_paralelo, max_por_ram)
paralelo <- as.integer(min(paralelo, length(semillas_vm)))
hilos <- max(1L, floor(nucleos / paralelo))

cat("RAM total:", round(ram_total_gb), "GB | nucleos:", nucleos, "\n")
cat("Semillas en esta VM:", length(semillas_vm), "| a la vez:", paralelo, "| hilos por corrida:", hilos, "\n")
if (paralelo < min(CONFIG$max_paralelo, length(semillas_vm)) && !CONFIG$forzar_paralelo)
  cat("ATENCION: recortado por RAM (", CONFIG$ram_por_corrida_gb, "GB por corrida ).",
      "Las semillas que no entran esperan en cola y arrancan solas cuando termina otra.\n")

RAM total: 499 GB | nucleos: 12 
Semillas en esta VM: 5 | a la vez: 1 | hilos por corrida: 12 


### Generar un script por semilla
Quedan en `~/exp6_control/<variante>/scripts` y se copian al bucket. Revisá la tabla: semilla → experimento → carpeta de salida.

In [4]:
dir_var     <- dir_variante()
dir_scripts <- file.path(dir_var, "scripts")
dir_logs    <- file.path(dir_var, "logs")
dir.create(dir_scripts, recursive = TRUE, showWarnings = FALSE)
dir.create(dir_logs, recursive = TRUE, showWarnings = FALSE)
dir_var <- normalizePath(dir_var); dir_scripts <- normalizePath(dir_scripts); dir_logs <- normalizePath(dir_logs)
dir_bucket <- file.path(CONFIG$carpeta_bucket, CONFIG$variante)
dir.create(dir_bucket, recursive = TRUE, showWarnings = FALSE)
if (!file.exists(CONFIG$notebook_base)) stop("No existe la notebook base: ", CONFIG$notebook_base)

corridas <- data.table(variante = CONFIG$variante, semilla = CONFIG$semillas,
                       experimento = CONFIG$experimento_inicial + seq_along(CONFIG$semillas) - 1L)
corridas <- corridas[semilla %in% semillas_vm]

# PROTECCION: frena si algun numero de experimento ya lo usa OTRA variante
otras <- list.files(CONFIG$carpeta_bucket, pattern = "^corridas\\.csv$", recursive = TRUE, full.names = TRUE)
otras <- otras[basename(dirname(otras)) != CONFIG$variante]
usados <- rbindlist(lapply(otras, function(f) fread(f, select = c("variante", "experimento"))))
if (nrow(usados) && any(corridas$experimento %in% usados$experimento)) {
  choque <- usados[experimento %in% corridas$experimento]
  stop("Los experimentos ", paste(unique(choque$experimento), collapse = ", "),
       " ya son de la variante '", paste(unique(choque$variante), collapse = "', '"),
       "'. Cambia experimento_inicial en CONFIG.", call. = FALSE)
}
corridas[, script := mapply(generar_script, semilla, experimento,
                            MoreArgs = list(cfg = CONFIG, dir_scripts = dir_scripts))]
corridas[, log := file.path(dir_logs, paste0(nombre_corrida(experimento, semilla), ".log"))]
corridas[, overrides := paste(names(CONFIG$param_override),
                              sapply(CONFIG$param_override, function(v) paste(deparse(v), collapse = "")),
                              sep = "=", collapse = "; ")]
corridas[, carpeta_salida := paste0("/content/buckets/b1/exp/WF", experimento)]
fwrite(corridas, file.path(dir_var, "corridas.csv"))
writeLines(toJSON(CONFIG, auto_unbox = TRUE, pretty = TRUE, null = "null"), file.path(dir_var, "CONFIG.json"))

# copia de configuracion y scripts al bucket (queda aunque se borre la VM)
file.copy(file.path(dir_var, c("corridas.csv", "CONFIG.json")), dir_bucket, overwrite = TRUE)
dir.create(file.path(dir_bucket, "scripts"), showWarnings = FALSE)
file.copy(corridas$script, file.path(dir_bucket, "scripts"), overwrite = TRUE)

# aviso si la carpeta de salida ya existe (choque de numero de experimento o corrida a retomar)
for (k in seq_len(nrow(corridas))) {
  cs <- corridas$carpeta_salida[k]
  if (dir.exists(cs) && length(list.files(cs)))
    cat("ATENCION:", cs, "ya tiene archivos. Si es un relanzamiento esta bien (retoma HT.RDATA y modelos/);",
        "si es OTRA variante, cambia experimento_inicial.\n")
}
corridas[, .(variante, semilla, experimento, carpeta_salida)]

[1] TRUE TRUE

[1] TRUE TRUE TRUE TRUE TRUE

ATENCION: /content/buckets/b1/exp/WF9540 ya tiene archivos. Si es un relanzamiento esta bien (retoma HT.RDATA y modelos/); si es OTRA variante, cambia experimento_inicial.


variante,semilla,experimento,carpeta_salida
<chr>,<dbl>,<int>,<chr>
boruta_trad,109843,9540,/content/buckets/b1/exp/WF9540
boruta_trad,109847,9541,/content/buckets/b1/exp/WF9541
boruta_trad,109849,9542,/content/buckets/b1/exp/WF9542
boruta_trad,109859,9543,/content/buckets/b1/exp/WF9543
boruta_trad,109873,9544,/content/buckets/b1/exp/WF9544


### Lanzar
Arranca todo en segundo plano. Si una corrida se cae (o se muere la VM spot), volvé a correr esta celda: solo relanza las que no terminaron OK y la base retoma desde `HT.RDATA` y `modelos/` (Boruta sí se vuelve a correr desde cero).

In [5]:
# Lanza las corridas en segundo plano (nohup): siguen aunque cierres el navegador
# o reinicies el kernel de esta notebook.
# Si lo volves a correr, solo lanza las que NO terminaron OK (la base retoma HT.RDATA y modelos/).

lanzador <- file.path(dir_var, "lanzador.sh")
if (nrow(procesos(lanzador)) > 0) stop("Ya hay un lanzador corriendo para esta variante. Mira estado() o usa detener_todo().")

est <- estado()
pendientes <- corridas[!(experimento %in% est[estado == "OK", experimento])]
if (nrow(pendientes) == 0) stop("Todas las corridas de esta variante ya terminaron OK.")

# conservo logs de intentos anteriores
for (lg in pendientes$log) if (file.exists(lg))
  file.rename(lg, paste0(lg, ".", format(Sys.time(), "%Y%m%d_%H%M%S"), ".prev"))

writeLines(pendientes$script, file.path(dir_var, "pendientes.txt"))
writeLines(c(
  "#!/bin/bash",
  paste0("cd ", shQuote(dir_var)),
  paste0("export OMP_NUM_THREADS=", hilos),
  paste0("export BUCKET_LOGS=", shQuote(file.path(dir_bucket, "logs"))),
  "mkdir -p \"$BUCKET_LOGS\"",
  "echo \"LANZADOR_INICIO $(date)\"",
  paste0("xargs -a ", shQuote(file.path(dir_var, "pendientes.txt")), " -P ", paralelo,
         " -I{} bash -c 's=\"{}\"; b=$(basename \"$s\" .R); echo \"arranca $b $(date)\";",
         " Rscript \"$s\" > \"logs/$b.log\" 2>&1; echo \"termina $b $(date)\";",
         " cp \"logs/$b.log\" \"$BUCKET_LOGS/\"'"),
  "echo \"LANZADOR_FIN $(date)\""
), lanzador)

system(paste0("setsid nohup bash ", shQuote(lanzador), " > ", shQuote(file.path(dir_var, "lanzador.log")), " 2>&1 &"))
Sys.sleep(5)
cat("Lanzadas", nrow(pendientes), "corridas,", paralelo, "a la vez\n")
estado()

08:01:33 | RAM VM: usada 5 GB de 499 GB, disponible 494 GB
Lanzadas 5 corridas, 1 a la vez
08:01:38 | RAM VM: usada 6 GB de 499 GB, disponible 493 GB


experimento,semilla,estado,ram_gb,ultima_celda,min_sin_escribir
<int>,<int>,<chr>,<dbl>,<chr>,<dbl>
9540,109843,corriendo,0.1,[celda 17/119] ram=0.1GB pico=0.1GB,0.1
9541,109847,pendiente,NA,,NA
9542,109849,pendiente,NA,,NA
9543,109859,pendiente,NA,,NA
9544,109873,pendiente,NA,,NA


### Monitor
`estado`: pendiente / corriendo / OK / ERROR / detenido. `ram_gb` es la RAM actual de cada corrida; en el log, cada celda muestra también el **pico**. Con el pico de la primera corrida ajustá `ram_por_corrida_gb`.

In [29]:
# Correr esta celda cuando quieras para ver como van
estado()

21:55:30 | RAM VM: usada 5 GB de 499 GB, disponible 493 GB


experimento,semilla,estado,ram_gb,ultima_celda,min_sin_escribir
<int>,<int>,<chr>,<dbl>,<chr>,<dbl>
9540,109843,OK,NA,[celda 119/119] ram=27.1GB pico=60GB,659.5
9541,109847,OK,NA,[celda 119/119] ram=29.7GB pico=58.6GB,504.0
9542,109849,OK,NA,[celda 119/119] ram=31GB pico=57.4GB,337.7
9543,109859,OK,NA,[celda 119/119] ram=29.8GB pico=58.5GB,181.5
9544,109873,OK,NA,[celda 119/119] ram=29.3GB pico=60GB,18.0


### Ver el log de una semilla

In [28]:
# Ultimas lineas del log de una semilla (durante Boruta: una linea por ronda)
ver_log(109859, n = 20)

100%|██████████| 1.73M/1.73M [00:00<00:00, 2.81MB/s]

  0%|          | 0.00/1.73M [00:00<?, ?B/s]
 70%|███████   | 1.22M/1.73M [00:00<00:00, 5.02MB/s]
100%|██████████| 1.73M/1.73M [00:00<00:00, 2.72MB/s]

  0%|          | 0.00/1.73M [00:00<?, ?B/s]
 73%|███████▎  | 1.27M/1.73M [00:00<00:00, 8.91MB/s]
100%|██████████| 1.73M/1.73M [00:00<00:00, 3.27MB/s]

[celda 118/119]  2026-11-07 18:53:58   ram=29.8GB pico=58.5GB 
Loading required package: yaml

[celda 119/119]  2026-11-07 18:53:58   ram=29.8GB pico=58.5GB 
[1] "Sat Nov 07 18:53:58 2026"

###FIN_OK###  2026-11-07 18:53:58   ram=29.8GB pico=58.5GB 


### Detener todo (solo si hace falta)

In [ ]:
# SOLO si queres cortar todo:
# detener_todo()